In [ ]:
import os
import pandas as pd
import mlflow
from mlxtend.feature_selection import SequentialFeatureSelector as SFS
from sklearn.neighbors import KNeighborsClassifier
from dotenv import load_dotenv

load_dotenv()

estimator = KNeighborsClassifier(n_neighbors=3)

os.environ["MLFLOW_S3_ENDPOINT_URL"] = os.getenv("MLFLOW_S3_ENDPOINT_URL")
os.environ["AWS_ACCESS_KEY_ID"] = os.getenv("AWS_ACCESS_KEY_ID")
os.environ["AWS_SECRET_ACCESS_KEY"] = os.getenv("AWS_SECRET_ACCESS_KEY")

initial_csv_path = "s3://s3-student-mle-20260614-fb07c65e05/19/9829f204cdb4482598e011bf36c51d85/artifacts/models_artifacts/data/initial_data.csv"
pipeline_initial_path = "s3://s3-student-mle-20260614-fb07c65e05/19/9829f204cdb4482598e011bf36c51d85/artifacts/models"

storage_options = {
    "key": os.getenv("AWS_ACCESS_KEY_ID"),
    "secret": os.getenv("AWS_SECRET_ACCESS_KEY"),
    "client_kwargs": {"endpoint_url": "https://storage.yandexcloud.net"},
}
df_new: pd.DataFrame = pd.read_csv(
    initial_csv_path, storage_options=storage_options
)
pipeline = mlflow.sklearn.load_model(pipeline_initial_path)
preprocessor = pipeline.named_steps["preprocessor"]
model = pipeline.named_steps["model"]

X = df_new.drop(columns=["target", "end_date"], errors="ignore")
y = df_new["target"]
X_trans = preprocessor.transform(X)

model = CatBoostClassifier(auto_class_weights=auto_class_weights)

array([0, 0, 1, ..., 0, 1, 0])

In [15]:
from util.test_load_data_from_db import load_data_from_db

df = load_data_from_db()

In [2]:
import optuna
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import Lasso
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# загружаем датасет
data = fetch_california_housing()
X, y = data.data, data.target

# разбиваем данные на тренировочный и тестовый наборы
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


# определяем функцию objective для Optuna
def objective(trial):
    # предлагаем гиперпараметры
    alpha = trial.suggest_float(
        "alpha", 1e-4, 10.0, log=True
    )  # L1-регуляризация

    # создаём и обучаем модель
    model = Lasso(alpha=alpha, random_state=0)
    model.fit(X_train, y_train)

    # предсказываем и вычисляем RMSE
    preds = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, preds))

    return rmse


# создаём study и проводим оптимизацию
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=10)

# выводим результаты
print("Number of finished trials:", len(study.trials))
print("Best trial:", study.best_trial.params)

[I 2026-07-22 20:19:30,424] A new study created in memory with name: no-name-5406d3c5-2b8d-413f-9c5a-62e5ce8b11dc
[I 2026-07-22 20:19:30,432] Trial 0 finished with value: 0.7429237843052565 and parameters: {'alpha': 0.0020881132234280425}. Best is trial 0 with value: 0.7429237843052565.
[I 2026-07-22 20:19:30,469] Trial 1 finished with value: 0.7393001010641351 and parameters: {'alpha': 0.01586989052977261}. Best is trial 1 with value: 0.7393001010641351.
[I 2026-07-22 20:19:30,524] Trial 2 finished with value: 0.7396470346397558 and parameters: {'alpha': 0.016962699558947348}. Best is trial 1 with value: 0.7393001010641351.
[I 2026-07-22 20:19:30,557] Trial 3 finished with value: 0.744158540590337 and parameters: {'alpha': 0.00106435750165304}. Best is trial 1 with value: 0.7393001010641351.
[I 2026-07-22 20:19:30,571] Trial 4 finished with value: 0.7450966716147942 and parameters: {'alpha': 0.0003509033019381921}. Best is trial 1 with value: 0.7393001010641351.
[I 2026-07-22 20:19:30

Number of finished trials: 10
Best trial: {'alpha': 0.01586989052977261}


In [ ]:
import optuna
import numpy as np
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# загружаем датасет
data = load_iris()
X, y = data.data, data.target

# разбиваем на тренировочный и тестовый наборы
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


# определяем функцию objective для Optuna
def objective(trial):
    # здесь предложите гиперпараметры модели случайного леса
    n_estimators = trial.suggest_int("n_estimators", 10, 200)
    max_depth = trial.suggest_int("max_depth", 2, 32, log=True)
    min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
    min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
    bootstrap = trial.suggest_categorical("bootstrap", [True, False])

    # создайте и обучите модель случайного леса с предложенными гиперпараметрами
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        bootstrap=bootstrap,
        random_state=42,
    )
    # предсказание и вычисление точности
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    accuracy = accuracy_score(y_test, preds)

    return accuracy  # верните значение точности


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=10)

# выведите результаты
print("Лучшие параметры:", study.best_trial.params)

[I 2026-07-22 20:22:44,439] A new study created in memory with name: no-name-661ac892-d4ec-4c5a-8623-32e16d38d097
[I 2026-07-22 20:22:44,447] Trial 0 finished with value: 1.0 and parameters: {'n_estimators': 11, 'max_depth': 20, 'min_samples_split': 8, 'min_samples_leaf': 7, 'bootstrap': False}. Best is trial 0 with value: 1.0.
[I 2026-07-22 20:22:44,482] Trial 1 finished with value: 1.0 and parameters: {'n_estimators': 68, 'max_depth': 2, 'min_samples_split': 16, 'min_samples_leaf': 18, 'bootstrap': True}. Best is trial 0 with value: 1.0.
[I 2026-07-22 20:22:44,501] Trial 2 finished with value: 1.0 and parameters: {'n_estimators': 42, 'max_depth': 4, 'min_samples_split': 3, 'min_samples_leaf': 11, 'bootstrap': True}. Best is trial 0 with value: 1.0.
[I 2026-07-22 20:22:44,557] Trial 3 finished with value: 1.0 and parameters: {'n_estimators': 154, 'max_depth': 2, 'min_samples_split': 11, 'min_samples_leaf': 12, 'bootstrap': True}. Best is trial 0 with value: 1.0.
[I 2026-07-22 20:22:44

Лучшие параметры: {'n_estimators': 11, 'max_depth': 20, 'min_samples_split': 8, 'min_samples_leaf': 7, 'bootstrap': False}


In [ ]:
import numpy as np
import torch
import tensorflow as tf
import random

# Numpy
np.random.seed(42)

# Torch
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

# Tensforflow, Keras
tf.random.set_seed(42)

# Python
random.seed(42)

In [11]:
import optuna
from optuna.samplers import CmaEsSampler
from optuna.integration.mlflow import MLflowCallback
from sklearn.ensemble import BaggingClassifier
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import numpy as np
import random

# загрузка датасета
digits = load_digits()
X, y = digits.data, digits.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


# фиксация seed
random.seed(42)
np.random.seed(42)

# настройка Optuna - изменение правила логирования optuna.logging.INFO
optuna.logging.set_verbosity(optuna.logging.INFO)

storage = "sqlite:///example.db"
study_name = "bagging-optimization-study"

# MLflow Callback
mlflc = MLflowCallback(
    tracking_uri="http://localhost:5003", metric_name="accuracy"
)
# mlflc = MLflowCallback(
#     tracking_uri="http://localhost:5003",
#     # experiment_name="bagging-optimization",
# )


def objective(trial):
    n_estimators = trial.suggest_int("n_estimators", 10, 200)
    max_samples = trial.suggest_float("max_samples", 0.1, 1.0)
    max_features = trial.suggest_float("max_features", 0.1, 1.0)

    clf = BaggingClassifier(
        n_estimators=n_estimators,
        max_samples=max_samples,
        max_features=max_features,
        random_state=42,
    )

    clf.fit(X_train, y_train)
    preds = clf.predict(X_test)
    accuracy = accuracy_score(y_test, preds)

    return accuracy


# создание study с CmaEsSampler и оптимизация
sampler = CmaEsSampler(seed=42)
study = optuna.create_study(
    study_name=study_name,
    storage=storage,
    sampler=sampler,
    direction="maximize",
    load_if_exists=True,
)
study.optimize(objective, n_trials=10, callbacks=[mlflc])

# вывод результатов (число итераций и лучшие гиперпараметры)
print("Number of finished trials: ", len(study.trials))
print("Best trial:", study.best_trial.params)

/var/folders/9p/vkzz84_d4_l8l4s8mjml37740000gn/T/ipykernel_27968/3093274573.py:30: FutureWarning: MLflowCallback has been deprecated in v4.9.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v4.9.0.
  mlflc = MLflowCallback(
[I 2026-07-22 21:01:51,625] Using an existing study with name 'bagging-optimization-study' instead of creating a new one.
[I 2026-07-22 21:01:52,018] Trial 12 finished with value: 0.9694444444444444 and parameters: {'n_estimators': 199, 'max_samples': 0.41790661836719145, 'max_features': 0.37806776870936276}. Best is trial 5 with value: 0.9777777777777777.


🏃 View run 12 at: http://localhost:5003/#/experiments/23/runs/19302a5c22684631a59945648615772f
🧪 View experiment at: http://localhost:5003/#/experiments/23


[I 2026-07-22 21:01:54,003] Trial 13 finished with value: 0.9722222222222222 and parameters: {'n_estimators': 95, 'max_samples': 0.5341544587640918, 'max_features': 0.6240534046236716}. Best is trial 5 with value: 0.9777777777777777.


🏃 View run 13 at: http://localhost:5003/#/experiments/23/runs/2643118361e6401ca38504385fd085f0
🧪 View experiment at: http://localhost:5003/#/experiments/23


[I 2026-07-22 21:01:55,694] Trial 14 finished with value: 0.9777777777777777 and parameters: {'n_estimators': 154, 'max_samples': 0.5768238475876591, 'max_features': 0.49944465877627386}. Best is trial 5 with value: 0.9777777777777777.


...

🏃 View run 19 at: http://localhost:5003/#/experiments/23/runs/6e8921af7f0b400ca13d0998d52611c6
🧪 View experiment at: http://localhost:5003/#/experiments/23


[I 2026-07-22 21:02:05,338] Trial 20 finished with value: 0.9777777777777777 and parameters: {'n_estimators': 179, 'max_samples': 0.5429769946522345, 'max_features': 0.4962207943490313}. Best is trial 5 with value: 0.9777777777777777.


🏃 View run 20 at: http://localhost:5003/#/experiments/23/runs/105703bb1671487e9af0eb75f7807b59
🧪 View experiment at: http://localhost:5003/#/experiments/23


[I 2026-07-22 21:02:06,852] Trial 21 finished with value: 0.9666666666666667 and parameters: {'n_estimators': 143, 'max_samples': 0.7812719375504188, 'max_features': 0.3607515810891331}. Best is trial 5 with value: 0.9777777777777777.


🏃 View run 21 at: http://localhost:5003/#/experiments/23/runs/e8d7f3f4af804307ae310c416431a9ca
🧪 View experiment at: http://localhost:5003/#/experiments/23
Number of finished trials:  22
Best trial: {'n_estimators': 162, 'max_samples': 0.7015977116089022, 'max_features': 0.5210077574107288}
